### TODO:

- [x] Create the preprocessing function
- [x] Get the marker genes
- [x] Intersect the single cell genes and the bulk genes
- [x] Select the highest r2 genes
- [x] Compare them to the signature marker genes

In [1]:
import scanpy as sc
from benchmark_utils import preprocess_scrna, load_bulk_facs
import pickle

with open("project/sc_dlbcl_genes_list_map.pkl", "rb") as f:
    genes_list_map = pickle.load(f)

In [2]:
def add_cell_types_grouped(adata, group):
    if group == "DLBCL_2nd_level_granularity":
        adata.obs["cell_types_grouped_DLBCL_2nd_level_granularity"] = adata.obs["celltype_level2_scanvi"]
        return adata, None
    else:
        raise ValueError(f"Group {group} not found")

In [3]:
adata = sc.read("/home/owkin/data/dataset-1df11f03-9587-49f9-a34d-cb15a4b72b6a/v2/dataset_level/dataset-1df11f03-9587-49f9-a34d-cb15a4b72b6a/merge_samples/sc_merged_annotated.h5ad")

In [ ]:
adata.obs.columns

In [5]:
adata_2 = sc.read_h5ad("/home/owkin/data/dataset-1df11f03-9587-49f9-a34d-cb15a4b72b6a/v3/dataset_level/dataset-1df11f03-9587-49f9-a34d-cb15a4b72b6a/merge_samples/sc_merged.h5ad")

In [ ]:
adata_2.obs.columns

In [4]:
adata = preprocess_scrna(adata, keep_genes=None, batch_key="donor_id")
adata, _ = add_cell_types_grouped(adata, "DLBCL_2nd_level_granularity")

single_cell_data = adata.to_df()

In [5]:
import pandas as pd

def load_bulk_dlbcl() -> dict:
    dlbcl_data = pd.read_csv("/home/owkin/data/dataset-6253ffa8-7098-4928-8323-21aeb644d19d/filtered_raw_counts.tsv", sep="\t", index_col=0)
    return {"dataset": dlbcl_data}

In [ ]:
pd.Series(genes_list_map.values()).value_counts()[:10]

In [ ]:
duplicated_genes_set = {"ENSG00000018408", "ENSG00000134107", "ENSG00000135698", "ENSG00000136628", "ENSG00000196535" }
duplicated_genes = {gene: ensembl_id for gene, ensembl_id in genes_list_map.items() if ensembl_id in duplicated_genes_set}
duplicated_genes

In [14]:
single_cell_data.drop(["DEC1", "EPRS", "MPP6", "TIAF1", "WWTR1"], axis=1, inplace=True)

In [15]:
single_cell_data.rename(columns=genes_list_map, inplace=True)

In [ ]:
bulk_dlbcl_data = load_bulk_dlbcl()
bulk_dlbcl_data = bulk_dlbcl_data["dataset"].T

bulk_dlbcl_data.head()

In [17]:
common_genes = list(set(bulk_dlbcl_data.columns) & set(single_cell_data.columns))

In [ ]:
single_cell_data[common_genes].shape

In [19]:
X = single_cell_data[common_genes].values

In [ ]:
len(common_genes)

In [ ]:
X.shape

In [22]:
group_codes, groups = pd.factorize(adata.obs["cell_types_grouped_DLBCL_2nd_level_granularity"].values,sort=True) 

In [23]:
sc_data = {}

for code, group in enumerate(groups):
    sc_data[group] = X[group_codes == code]

In [ ]:
sc_data.keys()

In [25]:
import numpy as np

def calculate_fisher_r2(X_clean, sc_data):
    SS_group = {}
    mu_group = {}
    n_k = {}

    for group in sc_data.keys():
        group_data = sc_data[group]
        SS_group[group] = np.var(group_data, axis=0)
        n_k[group] = group_data.shape[0]
        SS_group[group] = SS_group[group] * n_k[group]
        mu_group[group] = np.mean(group_data, axis=0)

    mu_total = np.mean(X_clean, axis=0)
    SS_between = np.zeros(X_clean.shape[1])

    for group in sc_data.keys():
        SS_between += n_k[group] * (mu_group[group] - mu_total)**2
        
    SS_within = np.sum([SS_group[group] for group in sc_data.keys()], axis=0)
    fisher_r2 = SS_between / (SS_between + SS_within)

    return fisher_r2

In [ ]:
sc_data["T_NK"].shape

In [28]:
fisher_r2 = calculate_fisher_r2(X, sc_data)

In [ ]:
fisher_r2_series = pd.Series(fisher_r2, index=common_genes)

fisher_r2_series.sort_values(ascending=False, inplace=True)

fisher_r2_series.head(100)

In [30]:
trial = set(fisher_r2_series[:2000].index.tolist()) & {"ENSG00000018408", "ENSG00000134107", "ENSG00000135698", "ENSG00000136628", "ENSG00000196535" }

In [ ]:
f"Just {len(trial)} genes in the final selection belong to the wrongly mapped ones."

In [ ]:
with open("project/highest_r2_genes_DLBCL_2nd_gran.pkl", "wb") as f:
    pickle.dump(fisher_r2_series[:2000].index.tolist(), f)

In [3]:
import pandas as pd

In [ ]:
signature_matrix = pd.read_csv("/home/owkin/data/dataset-1df11f03-9587-49f9-a34d-cb15a4b72b6a/v2/dataset_level/dataset-1df11f03-9587-49f9-a34d-cb15a4b72b6a/signature_matrices/signature_level2.csv", index_col=0)

signature_matrix.head()

In [5]:
signature_matrix.rename(genes_list_map, axis = 0, inplace=True)

In [ ]:
signature_matrix.head()

In [ ]:
for gene in signature_matrix.index:
    if not gene.startswith("ENSG"):
        print(gene)

signature_matrix.shape

In [15]:
extra_map = {
    "VNN3": "ENSG00000093134",
    "AC099489.1": "ENSG00000188897",
    "PCDHGA7": "ENSG00000253537",
    "PCDHGA6": "ENSG00000253731",
    "PCDHGA5": "ENSG00000253485",
    "FCGR1B": "ENSG00000198019",
}

In [ ]:
signature_matrix.rename(extra_map, axis = 0, inplace=True)

In [13]:
for gene in signature_matrix.index:
    if gene == "ENSG00000198019":
        print("the gene")



In [ ]:
signature_matrix.head()

In [20]:
signature_matrix.index.name = "Genes"

In [ ]:
signature_matrix.head()

In [19]:
#signature_matrix.to_csv("/home/owkin/project/data/dlbcl_data/signture_matrix_level2_ensg.csv")

In [38]:
marker_genes = signature_matrix.index.tolist()

In [ ]:
len(marker_genes)

In [43]:
marker_genes_list = [genes_list_map[gene] for gene in marker_genes]

In [ ]:
print(f"The overlap between the signature genes and the ones that we have selected is {round((len(set(marker_genes_list) & set(fisher_r2_series[:2000].index.tolist())))/len(set(marker_genes_list))*100, 2)}%")

In [12]:
from benchmark_utils import load_bulk_facs, load_dlbcl_sc

In [2]:
facs_bulk_data = load_bulk_facs()

In [ ]:
facs_bulk_data.keys()

In [ ]:
facs_bulk_data["dataset"].shape

In [ ]:
import pandas as pd

dlbcl_data = pd.read_csv("/home/owkin/data/dataset-6253ffa8-7098-4928-8323-21aeb644d19d/filtered_raw_counts.tsv", sep="\t", index_col=0)

dlbcl_data.shape


In [ ]:
dlbcl_data.head()

In [13]:
adata_dlbcl_sc = load_dlbcl_sc()

In [ ]:
adata_dlbcl_sc["dataset"].obs.columns

In [ ]:
adata_dlbcl_sc["dataset"].obs.celltype_level1_scanvi.value_counts()

In [ ]:
adata_dlbcl_sc["dataset"].obs.celltype_level2_scanvi.value_counts()

In [19]:
ground_truth_dlbcl_bulk = pd.read_csv("/home/owkin/data/dataset-6253ffa8-7098-4928-8323-21aeb644d19d/deconvolution_mcpcounter.tsv", sep = "\t", index_col=0)

In [ ]:
ground_truth_dlbcl_bulk

In [ ]:
ground_truth_dlbcl_bulk.index

In [28]:
proportions = ground_truth_dlbcl_bulk / ground_truth_dlbcl_bulk.sum(axis=0)

In [33]:
from scipy.special import softmax
# Apply softmax along axis 0 to get proportions that sum to 1
softmaxed_proportions = pd.DataFrame(
    softmax(ground_truth_dlbcl_bulk.values, axis=0),
    index=ground_truth_dlbcl_bulk.index,
    columns=ground_truth_dlbcl_bulk.columns
)


In [ ]:
softmaxed_proportions

In [ ]:
proportions

In [ ]:
proportions.sum(axis=0)

In [17]:
def load_dlbcl_bulk():
    bulk_data = pd.read_csv("/home/owkin/data/dataset-6253ffa8-7098-4928-8323-21aeb644d19d/filtered_raw_counts.tsv", sep="\t", index_col=0)
    ground_truth_data = pd.read_csv("/home/owkin/data/dataset-6253ffa8-7098-4928-8323-21aeb644d19d/deconvolution_mcpcounter.tsv", sep = "\t", index_col=0)
    #change the label names in the ground truth data as this is just a dummy dataset, not a real one
    signature_to_celltype = {
        'T cells': 'Malignant_dlbcl',
        'CD8 T cells': 'T_NK', 
        'Cytotoxic lymphocytes': 'Plasma',
        'B lineage': 'Mast',
        'NK cells': 'Epithelial',
        'Monocytic lineage': 'MoMac',
        'Myeloid dendritic cells': 'DC',
        'Neutrophils': 'Granulocyte',
        'Endothelial cells': 'Endothelial',
        'Fibroblasts': 'Fibroblast'
    }
    ground_truth_data.rename(signature_to_celltype, axis = 0, inplace=True)
    return {"dataset": bulk_data, "ground_truth": ground_truth_data.T}

In [9]:
from benchmark_utils import load_bulk_facs
import pandas as pd

In [10]:
right_example = load_bulk_facs()

In [18]:
trial_example = load_dlbcl_bulk()

In [ ]:
right_example.keys(), trial_example.keys()

In [ ]:
right_example["dataset"].shape, trial_example["dataset"].shape

In [ ]:
right_example["ground_truth"].shape, trial_example["ground_truth"].shape

In [ ]:
trial_example["ground_truth"].head()

In [1]:
from benchmark_utils import load_dlbcl_sc

In [2]:
trial = load_dlbcl_sc()

In [ ]:
trial["dataset"].to_df().head()

In [3]:
trial["dataset"].var_names

Index(['ENSG00000148584', 'ENSG00000175899', 'ENSG00000166535',
       'ENSG00000184389', 'ENSG00000128274', 'ENSG00000094914',
       'ENSG00000081760', 'ENSG00000114771', 'ENSG00000197953',
       'ENSG00000109576',
       ...
       'ENSG00000175535', 'ENSG00000147596', 'ENSG00000166091',
       'ENSG00000186867', 'ENSG00000121871', 'ENSG00000182601',
       'ENSG00000188624', 'ENSG00000188691', 'ENSG00000189139',
       'ENSG00000185888'],
      dtype='object', length=16480)

In [32]:
trial["dataset"].var_names = trial["dataset"].var_names.map(lambda x: genes_list_map[x])

In [ ]:
trial["dataset"].var_names

In [34]:
trial["dataset"].write_h5ad("/home/owkin/project/data/dlbcl_data/dlbcl_sc_processed.h5ad")

In [35]:
trial_2 = load_dlbcl_sc()

In [ ]:
trial_2["dataset"].var_names